In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix,classification_report,f1_score,roc_auc_score

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
df=pd.read_csv('/Users/siva/Downloads/digital_marketing_campaign_dataset.csv')

In [5]:
df.head()

,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool,Conversion
0,8000,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,9,4,688,IsConfid,ToolConfid,1
1,8001,69,Male,41760,Email,Retention,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,7,2,3459,IsConfid,ToolConfid,1
2,8002,46,Female,88456,PPC,Awareness,1546.429596,0.277490,0.076423,2,8.223619,13.794901,0,11,2,8,2337,IsConfid,ToolConfid,1
3,8003,32,Female,44085,PPC,Conversion,539.525936,0.137611,0.088004,47,4.540939,14.688363,89,2,2,0,2463,IsConfid,ToolConfid,1
4,8004,60,Female,83964,PPC,Conversion,1678.043573,0.252851,0.109940,0,2.046847,13.993370,6,6,6,8,4345,IsConfid,ToolConfid,1


In [6]:
# Train test split

indep = df.drop('Conversion', axis=1)
dep = df['Conversion']

In [7]:
indep.head()

,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool
0,8000,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,9,4,688,IsConfid,ToolConfid
1,8001,69,Male,41760,Email,Retention,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,7,2,3459,IsConfid,ToolConfid
2,8002,46,Female,88456,PPC,Awareness,1546.429596,0.277490,0.076423,2,8.223619,13.794901,0,11,2,8,2337,IsConfid,ToolConfid
3,8003,32,Female,44085,PPC,Conversion,539.525936,0.137611,0.088004,47,4.540939,14.688363,89,2,2,0,2463,IsConfid,ToolConfid
4,8004,60,Female,83964,PPC,Conversion,1678.043573,0.252851,0.109940,0,2.046847,13.993370,6,6,6,8,4345,IsConfid,ToolConfid


In [9]:
dep.head()

0    1
1    1
2    1
3    1
4    1
Name: Conversion, dtype: int64

In [10]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical variables before scaling
# Create label encoders for categorical columns
label_encoders = {}

# Identify categorical columns (assuming they contain string/object data)
categorical_columns = indep.select_dtypes(include=['object']).columns

# Encode each categorical column
indep_encoded = indep.copy()
for column in categorical_columns:
    le = LabelEncoder()
    indep_encoded[column] = le.fit_transform(indep[column])
    label_encoders[column] = le  # Store encoder for future use

# train_test_split data with encoded features
X_train, X_test, y_train, y_test = train_test_split(indep_encoded, dep, test_size=0.2, random_state=0)

# Standard scaler - now works with numeric data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
#Smote for data imbalance
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print("Before SMOTE:\n", y_train.value_counts())
print("After SMOTE:\n", y_train_smote.value_counts())

Before SMOTE:
 Conversion
1    5606
0     794
Name: count, dtype: int64
After SMOTE:
 Conversion
1    5606
0    5606
Name: count, dtype: int64


In [12]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
param_grid={'n_neighbors': [4,7], 'metric' : ['minkowski'], 'p' : [2]}
grid=GridSearchCV(KNeighborsClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
grid.fit(X_train_smote, y_train_smote) 

Fitting 5 folds for each of 2 candidates, totalling 10 fits


,estimator,KNeighborsClassifier()
,param_grid,"{'metric': ['minkowski'], 'n_neighbors': [4, 7], 'p': [2]}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,7


In [13]:
re=grid.cv_results_
grid_predictions = grid.predict(X_test_scaled) 
#confusion matrix
cm = confusion_matrix(y_test, grid_predictions)
#classification_report
clf_report = classification_report(y_test, grid_predictions)
#f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
#roc_auc_score
roc_score = roc_auc_score(y_test,grid.predict_proba(X_test_scaled)[:,1])

In [14]:
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("\nThe confusion Matrix:\n",cm)
print("\nThe report:\n",clf_report)
print("\n ROC_AUC_Score:\n",roc_score)

The f1_macro value for best parameter {'metric': 'minkowski', 'n_neighbors': 7, 'p': 2}: 0.7683264533883728

The confusion Matrix:
 [[ 122   72]
 [ 368 1038]]

The report:
               precision    recall  f1-score   support

           0       0.25      0.63      0.36       194
           1       0.94      0.74      0.83      1406

    accuracy                           0.72      1600
   macro avg       0.59      0.68      0.59      1600
weighted avg       0.85      0.72      0.77      1600


 ROC_AUC_Score:
 0.7524013432857708
